# Preprocesamiento

## 1. Importar paquetes

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import RobustScaler, TargetEncoder
import joblib

## 2. Carga de los datos

In [2]:
repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "README.md").exists():
    repo_root = repo_root.parent

if not (repo_root / "README.md").exists():
    raise FileNotFoundError("No se encontro la raiz del repositorio (README.md).")

datos = 'trabajo_resultado_calidad.pickle'
ruta_completa = repo_root / '02_Datos' / '03_Trabajo' / datos
df = pd.read_pickle(ruta_completa)

print(f'Datos cargados desde: {ruta_completa}')
print(f'Forma del dataset: {df.shape}')
df.head()

Datos cargados desde: C:\Github\Asteroid_Classification\02_Datos\03_Trabajo\trabajo_resultado_calidad.pickle
Forma del dataset: (88292, 35)


,spkid,full_name,pha,H,diameter,albedo,diameter_sigma,e,a,q,...,sigma_i,sigma_om,sigma_w,sigma_ma,sigma_ad,sigma_n,sigma_tp,sigma_per,class,rms
0,2000001,' 1 Ceres',N,3.4,939.400,0.0900,0.200,0.077557,2.767657,2.553006,...,4.613200e-09,6.176900e-08,6.618400e-08,7.355900e-09,1.115900e-11,1.201400e-12,3.686700e-08,9.439100e-09,MBA,0.40633
4,2000005,' 5 Astraea',N,6.9,106.699,0.2740,3.140,0.190913,2.574037,2.082619,...,2.743600e-06,2.926000e-05,3.016900e-05,8.703600e-06,4.902000e-09,5.724700e-10,3.640700e-05,3.618200e-06,MBA,0.51439
6,2000007,' 7 Iris',N,5.6,199.830,0.2766,10.000,0.230145,2.387375,1.837933,...,2.582500e-06,2.641200e-05,2.707500e-05,7.014700e-06,2.469500e-09,3.370100e-10,2.662700e-05,1.699400e-06,MBA,0.38128
7,2000008,' 8 Flora',N,6.5,147.491,0.2260,1.025,0.155833,2.201415,1.858362,...,3.319600e-06,2.499000e-05,2.735000e-05,1.247600e-05,2.900800e-09,5.160100e-10,4.147700e-05,2.040100e-06,MBA,0.52919
8,2000009,' 9 Metis',N,6.3,190.000,0.1180,0.321,0.123300,2.386189,2.091972,...,2.009400e-06,2.425100e-05,2.609600e-05,1.049400e-05,3.524500e-09,5.273900e-10,3.918200e-05,2.655500e-06,MBA,0.43435


## 3. Validacion inicial de variables

Este primer paso valida consistencia basica del dataset antes de homogeneizar variables: tipos, nulos, cardinalidad y dispersion inicial.

In [3]:
# 3.1 Tipos de datos y resumen general
resumen_tipos = (
    df.dtypes.astype(str)
    .rename('dtype')
    .to_frame()
)

resumen_tipos['nulos'] = df.isna().sum()
resumen_tipos['pct_nulos'] = (resumen_tipos['nulos'] / len(df) * 100).round(3)
resumen_tipos['n_unicos'] = df.nunique(dropna=False)

print(f'Registros: {len(df):,} | Variables: {df.shape[1]}')
resumen_tipos.sort_values(['dtype', 'pct_nulos'], ascending=[True, False]).head(20)

Registros: 88,292 | Variables: 35


,dtype,nulos,pct_nulos,n_unicos
H,float64,0,0.0,169
diameter,float64,0,0.0,15124
albedo,float64,0,0.0,955
diameter_sigma,float64,0,0.0,2795
e,float64,0,0.0,88292
a,float64,0,0.0,88292
q,float64,0,0.0,88292
i,float64,0,0.0,88292
om,float64,0,0.0,88292
w,float64,0,0.0,88292


In [4]:
# 3.2 Variables con posibles problemas de consistencia
objetos = resumen_tipos[resumen_tipos['dtype'] == 'object']
altos_nulos = resumen_tipos[resumen_tipos['pct_nulos'] > 0]

print('Variables tipo object:')
display(objetos)

print('Variables con nulos:')
display(altos_nulos.sort_values('pct_nulos', ascending=False))

Variables tipo object:


,dtype,nulos,pct_nulos,n_unicos
full_name,object,0,0.0,88292
pha,object,0,0.0,2
class,object,0,0.0,11


Variables con nulos:


,dtype,nulos,pct_nulos,n_unicos


In [5]:
# 3.3 Dispersion inicial de variables numericas (base para homogeneizacion)
num_cols = df.select_dtypes(include=['number']).columns

resumen_numericas = df[num_cols].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T
resumen_numericas['rango'] = resumen_numericas['max'] - resumen_numericas['min']
resumen_numericas['iqr'] = resumen_numericas['75%'] - resumen_numericas['25%']

# Coeficiente de variacion aproximado para medir heterogeneidad de escala
resumen_numericas['cv_pct'] = (
    (resumen_numericas['std'] / resumen_numericas['mean'].replace(0, pd.NA)) * 100
).round(2)

resumen_numericas[['mean', 'std', 'min', '1%', '25%', '50%', '75%', '99%', 'max', 'rango', 'iqr', 'cv_pct']].sort_values('cv_pct', ascending=False).head(20)

,mean,std,min,1%,25%,50%,75%,99%,max,rango,iqr,cv_pct
sigma_per,1.451289e-03,3.213364e-01,9.439100e-09,4.997266e-06,1.180500e-05,1.896300e-05,3.055450e-05,1.974853e-04,9.474600e+01,9.474600e+01,1.874950e-05,22141.45
sigma_ad,6.148522e-07,1.219626e-04,1.115900e-11,7.034148e-09,1.541075e-08,2.342700e-08,3.656100e-08,1.957527e-07,3.574800e-02,3.574800e-02,2.115025e-08,19836.08
sigma_a,3.877699e-07,7.257304e-05,1.035600e-11,6.162600e-09,1.339875e-08,2.056650e-08,3.225300e-08,1.738835e-07,2.123000e-02,2.123000e-02,1.885425e-08,18715.49
sigma_tp,2.545417e-04,1.496514e-02,3.686700e-08,3.028364e-05,7.292775e-05,1.148300e-04,1.981125e-04,1.454100e-03,4.440300e+00,4.440300e+00,1.251848e-04,5879.25
sigma_q,1.958960e-07,3.306094e-06,1.947000e-11,6.827619e-08,1.105600e-07,1.418500e-07,1.905200e-07,7.255834e-07,9.040900e-04,9.040900e-04,7.996000e-08,1687.68
sigma_e,6.362810e-08,3.714860e-07,4.807000e-12,2.926400e-08,4.086100e-08,4.983100e-08,6.448250e-08,2.236245e-07,9.598200e-05,9.598200e-05,2.362150e-08,583.84
sigma_ma,3.888396e-05,1.099415e-04,7.355900e-09,7.670591e-06,1.660900e-05,2.432400e-05,3.958350e-05,2.438972e-04,1.967900e-02,1.967899e-02,2.297450e-05,282.74
sigma_om,5.842985e-05,1.240589e-04,6.176900e-08,1.022491e-05,2.304475e-05,3.502150e-05,6.137075e-05,3.686496e-04,1.144000e-02,1.143994e-02,3.832600e-05,212.32
sigma_n,2.926238e-09,6.058489e-09,1.201400e-12,1.121900e-09,1.762700e-09,2.283850e-09,3.213400e-09,1.065972e-08,8.751600e-07,8.751588e-07,1.450700e-09,207.04
sigma_w,7.818983e-05,1.582236e-04,6.618400e-08,1.713246e-05,3.507775e-05,5.126350e-05,8.274700e-05,4.626656e-04,1.843200e-02,1.843193e-02,4.766925e-05,202.36


## 4. Homogeneizacion de tipos y formatos

En este paso se estandarizan formatos de texto, se convierten columnas numericas al tipo correcto y se normalizan variables categoricas para dejar una base consistente para modelado.

In [6]:
# 4.1 Estandarizar formatos de texto
if 'df' not in globals():
    raise NameError("Primero ejecuta la celda '2. Carga de los datos' para definir df.")

df_h = df.copy()
dtypes_antes = df_h.dtypes.astype(str)

# Normalizacion de espacios y tokens de nulo en columnas tipo texto
obj_cols = df_h.select_dtypes(include=['object']).columns
for col in obj_cols:
    df_h[col] = (
        df_h[col]
        .astype(str)
        .str.strip()
        .replace({'?': pd.NA, '': pd.NA, 'nan': pd.NA, 'None': pd.NA})
    )

# Ajustes especificos de formato
if 'full_name' in df_h.columns:
    df_h['full_name'] = df_h['full_name'].str.replace("'", '', regex=False).str.strip()

if 'pha' in df_h.columns:
    df_h['pha'] = df_h['pha'].str.upper().str.strip()

if 'class' in df_h.columns:
    df_h['class'] = df_h['class'].str.upper().str.strip()

print('Columnas de texto normalizadas:', len(obj_cols))

Columnas de texto normalizadas: 3


In [7]:
# 4.2 Homogeneizar columnas numericas
# Columnas esperadas como numericas segun el diccionario del proyecto
numeric_cols = [
    'spkid', 'H', 'diameter', 'albedo', 'diameter_sigma',
    'e', 'a', 'q', 'i', 'om', 'w', 'ma', 'ad', 'n', 'tp', 'tp_cal',
    'per', 'per_y', 'moid', 'moid_ld',
    'sigma_e', 'sigma_a', 'sigma_q', 'sigma_i', 'sigma_om', 'sigma_w',
    'sigma_ma', 'sigma_ad', 'sigma_n', 'sigma_tp', 'sigma_per', 'rms'
]

numeric_cols = [c for c in numeric_cols if c in df_h.columns]

for col in numeric_cols:
    df_h[col] = pd.to_numeric(df_h[col], errors='coerce')

# spkid se mantiene como entero nullable para preservar faltantes si aparecen
if 'spkid' in df_h.columns:
    df_h['spkid'] = df_h['spkid'].round().astype('Int64')

print('Columnas numericas convertidas:', len(numeric_cols))

Columnas numericas convertidas: 32


In [7]:
# 4.3 Convertir categoricas a tipo category
cat_cols = [c for c in ['pha', 'class'] if c in df_h.columns]
for col in cat_cols:
    df_h[col] = df_h[col].astype('category')

print('Columnas categoricas convertidas:', cat_cols)
df_h[cat_cols].dtypes if cat_cols else 'No hay columnas categoricas objetivo.'

Columnas categoricas convertidas: ['pha', 'class']


pha      category
class    category
dtype: object

In [8]:
# 4.4 Resumen de homogeneizacion
resumen_dtypes = pd.DataFrame({
    'dtype_antes': dtypes_antes,
    'dtype_despues': df_h.dtypes.astype(str)
})
resumen_dtypes['cambio_tipo'] = resumen_dtypes['dtype_antes'] != resumen_dtypes['dtype_despues']

resumen_nulos = pd.DataFrame({
    'nulos_despues': df_h.isna().sum(),
    'pct_nulos_despues': (df_h.isna().sum() / len(df_h) * 100).round(3)
}).sort_values('pct_nulos_despues', ascending=False)

print('Variables con cambio de tipo:')
display(resumen_dtypes[resumen_dtypes['cambio_tipo']].sort_index())

print('Top variables con nulos despues de homogeneizar:')
display(resumen_nulos.head(15))

Variables con cambio de tipo:


,dtype_antes,dtype_despues,cambio_tipo
class,object,category,True
full_name,object,str,True
pha,object,category,True


Top variables con nulos despues de homogeneizar:


,nulos_despues,pct_nulos_despues
spkid,0,0.0
full_name,0,0.0
pha,0,0.0
H,0,0.0
diameter,0,0.0
albedo,0,0.0
diameter_sigma,0,0.0
e,0,0.0
a,0,0.0
q,0,0.0


## 5. Eliminación de identificadores

Se eliminan `spkid` y `full_name` del dataset. Ninguno aporta información predictiva: `spkid` es un índice de catálogo numérico sin significado físico, y `full_name` es texto libre que sklearn no puede procesar.

In [ ]:
# 5.1 Eliminar columnas identificadoras
cols_drop = [c for c in ['spkid', 'full_name'] if c in df_h.columns]
df_h = df_h.drop(columns=cols_drop)

print(f'Columnas eliminadas: {cols_drop}')
print(f'Dimensiones resultantes: {df_h.shape}')
df_h.dtypes

## 6. Encoding de variables categóricas

Se aplican dos estrategias según el rol de cada variable:

- **`pha` (target)**: mapeo binario directo `Y → 1`, `N → 0`.
- **`class` (feature)**: TargetEncoding. Reemplaza cada clase de asteroide (MBA, APO, etc.) por la proporción suavizada de PHAs que contiene, produciendo un valor `float64` entre 0 y 1. Al quedar como `float64`, esta columna será incluida automáticamente en el escalado de la sección siguiente.

In [ ]:
# 6.1 Mapeo binario del target
df_h['pha'] = df_h['pha'].map({'Y': 1, 'N': 0}).astype(int)

print('pha codificada:')
print(df_h['pha'].value_counts())

# 6.2 TargetEncoding de class
# Cada categoria se reemplaza por la proporcion suavizada de PHA=1 en esa clase
encoder = TargetEncoder(target_type='binary', smooth='auto', random_state=42)
df_h['class'] = encoder.fit_transform(df_h[['class']].astype(str), df_h['pha']).flatten()

print(f'\nclass codificada (proporcion de PHA por clase):')
print(df_h['class'].describe().round(4))

## 7. Escalado de variables numéricas

Se aplica `RobustScaler` sobre todas las variables `float64`. Al ejecutarse después del encoding, la columna `class` ya es `float64` y queda incluida en el escalado junto al resto de variables numéricas. El scaler ajustado se serializa para garantizar que en predicción se apliquen exactamente las mismas transformaciones.

In [ ]:
# 7.1 Seleccionar columnas a escalar: float64 (incluye class tras el encoding, excluye pha int)
cols_escalar = df_h.select_dtypes(include=['float64']).columns.tolist()

# 7.2 Ajustar y transformar
scaler = RobustScaler()
df_h[cols_escalar] = scaler.fit_transform(df_h[cols_escalar])

print(f'Variables escaladas: {len(cols_escalar)}')
print(cols_escalar)

# 7.3 Verificar resultado: mediana debe ser ~0 tras el escalado
df_h[cols_escalar].describe().T[['50%', 'mean', 'std']].round(4)

## 8. Selección de variables

Se eliminan las variables redundantes o sin valor predictivo independiente identificadas por el Astro Físico:

- `moid_ld`: unidades distintas de `moid`, correlación perfecta (r = 1.0)
- `ad`: afelio, derivada de `a` y `e` ($Q = a(1+e)$), irrelevante para acercamiento a Tierra
- `per`: período orbital, derivada de `a` ($P \propto a^{3/2}$), sin información adicional
- `per_y`: `per` en años, misma información en unidades distintas
- `n`: movimiento medio, inversa de `per`, misma información

In [ ]:
# 8.1 Eliminar variables redundantes
cols_redundantes = [c for c in ['moid_ld', 'ad', 'per', 'per_y', 'n'] if c in df_h.columns]
df_h = df_h.drop(columns=cols_redundantes)

print(f'Variables eliminadas: {cols_redundantes}')
print(f'Dimensiones resultantes: {df_h.shape}')
df_h.columns.tolist()

### 8.2 Generación de datasets finales

Se generan dos versiones del dataset preprocesado para los dos experimentos de entrenamiento:

- **`trabajo_preprocesado_moid.pickle`**: incluye `moid`. Modelo A — máximo poder predictivo, con posible dependencia del criterio de clasificación PHA.
- **`trabajo_preprocesado.pickle`**: excluye `moid`. Modelo B — variables físicas puras, sin información que define la etiqueta target.

In [ ]:
# 8.2 Guardar datasets finales

# Modelo A: incluye moid
ruta_moid = repo_root / '02_Datos' / '03_Trabajo' / 'trabajo_preprocesado_moid.pickle'
df_h.to_pickle(ruta_moid)
print(f'Dataset con moid guardado en: {ruta_moid}')

# Modelo B: sin moid
df_sin_moid = df_h.drop(columns=['moid']) if 'moid' in df_h.columns else df_h.copy()
ruta_sin_moid = repo_root / '02_Datos' / '03_Trabajo' / 'trabajo_preprocesado.pickle'
df_sin_moid.to_pickle(ruta_sin_moid)
print(f'Dataset sin moid guardado en: {ruta_sin_moid}')

print(f'\nDimensiones Modelo A (con moid):  {df_h.shape}')
print(f'Dimensiones Modelo B (sin moid):  {df_sin_moid.shape}')

## 9. Serialización de artefactos del pipeline

Se serializan el scaler y el encoder ajustados sobre los datos de trabajo. Estos artefactos se cargan en predicción para aplicar exactamente las mismas transformaciones a datos nuevos, garantizando consistencia con el entrenamiento.

In [ ]:
# Guardar artefactos del pipeline de preprocesamiento
ruta_scaler = repo_root / '04_Modelos' / 'pipeline_scaler.joblib'
joblib.dump(scaler, ruta_scaler)
print(f'Scaler guardado en: {ruta_scaler}')

ruta_encoder = repo_root / '04_Modelos' / 'pipeline_encoder.joblib'
joblib.dump(encoder, ruta_encoder)
print(f'Encoder guardado en: {ruta_encoder}')